# Reproducible Data Preprocessing

This notebook demonstrates Phase 4 checks and calls the reusable pipeline in `ml/src/preprocess.py` for the actual transformation. The raw CSV is read-only. Only `rv1` and `rv2` are removed: they are identical random/questionable variables without physical interpretation.

Statistical outliers are retained because unusually high energy consumption may represent legitimate building operation rather than erroneous measurements. No imputation, feature engineering, scaling, splitting, or modeling is performed.

In [1]:
from pathlib import Path
import sys
import pandas as pd

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidate_roots if (p / 'data/raw/energydata_complete.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate data/raw/energydata_complete.csv')
sys.path.insert(0, str(PROJECT_ROOT))
from ml.src.preprocess import preprocess_dataset, sha256_file
RAW_PATH = PROJECT_ROOT / 'data/raw/energydata_complete.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data/processed/energydata_cleaned.csv'
raw_checksum_before = sha256_file(RAW_PATH)
raw_data = pd.read_csv(RAW_PATH)
print(f'Initial raw shape: {raw_data.shape}')
print(f'Columns: {raw_data.columns.tolist()}')
print(f'Raw SHA-256 before preprocessing: {raw_checksum_before}')

Initial raw shape: (19735, 29)
Columns: ['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2']
Raw SHA-256 before preprocessing: 2820bf712ad0275cb18b85a05250926100d8e65ebb9f4d2d016ca91ea152a25d


## Data Quality and Date Handling

Date parsing is in memory. Integrity problems fail explicitly; this phase does not impute or silently delete records. The same checks are enforced again by the imported pipeline before it writes output.

In [2]:
required_columns = {'date', 'Appliances', 'rv1', 'rv2'}
missing_required = sorted(required_columns.difference(raw_data.columns))
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')
raw_data['date'] = pd.to_datetime(raw_data['date'], errors='raise')
missing_value_count = int(raw_data.isna().sum().sum())
duplicate_row_count = int(raw_data.duplicated().sum())
duplicate_timestamp_count = int(raw_data['date'].duplicated().sum())
timestamps_ordered = bool(raw_data['date'].is_monotonic_increasing)
print(f'Datetime dtype: {raw_data["date"].dtype}')
print(f'Missing values: {missing_value_count}')
print(f'Duplicate rows: {duplicate_row_count}')
print(f'Duplicate timestamps: {duplicate_timestamp_count}')
print(f'Chronologically ordered: {timestamps_ordered}')
assert missing_value_count == duplicate_row_count == duplicate_timestamp_count == 0
assert timestamps_ordered

Datetime dtype: datetime64[us]
Missing values: 0
Duplicate rows: 0
Duplicate timestamps: 0
Chronologically ordered: True


## Column Decision and Outliers

`rv1` and `rv2` are dropped because they are identical row by row and represent random/questionable rather than physical building variables. No other predictor, including `lights`, is removed. The IQR-flagged target values are kept: statistical outliers are retained because unusually high energy consumption may represent legitimate building operation rather than erroneous measurements.

In [3]:
report = preprocess_dataset(RAW_PATH, OUTPUT_PATH)
processed = pd.read_csv(OUTPUT_PATH, parse_dates=['date'], float_precision='round_trip')
print(f'Processed shape: {processed.shape}')
print(f'Columns removed: {report["removed_columns"]}')
print(f'Rows unchanged: {report["rows_unchanged"]}')
print(f'Processed CSV SHA-256: {report["processed_checksum"]}')
print(f'Raw SHA-256 after preprocessing: {sha256_file(RAW_PATH)}')

Processed shape: (19735, 27)
Columns removed: ['rv1', 'rv2']
Rows unchanged: True
Processed CSV SHA-256: d4d5964a61a5a6ddfc1148012abb77f14f826dfa0ebd6f9b5e52d93a6c705a19
Raw SHA-256 after preprocessing: 2820bf712ad0275cb18b85a05250926100d8e65ebb9f4d2d016ca91ea152a25d


In [4]:
print('Final columns:')
print(processed.columns.tolist())
print('Final data types:')
display(processed.dtypes)
assert len(processed) == len(raw_data)
assert processed.shape[1] == raw_data.shape[1] - 2
assert not {'rv1', 'rv2'}.intersection(processed.columns)
assert {'Appliances', 'date', 'lights'}.issubset(processed.columns)
assert processed.isna().sum().sum() == 0
assert processed.duplicated().sum() == 0
assert processed['date'].duplicated().sum() == 0
assert processed['date'].is_monotonic_increasing
assert report['raw_checksum_before'] == report['raw_checksum_after'] == raw_checksum_before
print('Rows unchanged; integrity checks passed; raw file checksum unchanged.')

Final columns:
['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint']
Final data types:


date           datetime64[us]
Appliances              int64
lights                  int64
T1                    float64
RH_1                  float64
T2                    float64
RH_2                  float64
T3                    float64
RH_3                  float64
T4                    float64
RH_4                  float64
T5                    float64
RH_5                  float64
T6                    float64
RH_6                  float64
T7                    float64
RH_7                  float64
T8                    float64
RH_8                  float64
T9                    float64
RH_9                  float64
T_out                 float64
Press_mm_hg           float64
RH_out                float64
Windspeed             float64
Visibility            float64
Tdewpoint             float64
dtype: object

Rows unchanged; integrity checks passed; raw file checksum unchanged.
